# Mixture of Experts (MoE) Fine-Tuning

## Overview
Mixture of Experts (MoE) architecture uses sparse activation to scale model capacity without proportional compute increase. This notebook covers MoE principles, router training, fine-tuning strategies, and practical applications.

**Key Topics:**
- MoE architecture fundamentals
- Router training and load balancing
- Expert specialization
- Sparse activation mechanics
- Fine-tuning MoE models (Mixtral, DeepSeek)
- Memory and compute trade-offs
- When to use MoE

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
from collections import defaultdict

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Historical Context & Evolution

### The Journey to Mixture of Experts

**1991 - Origins:**
- Jacobs et al.: "Adaptive Mixtures of Local Experts"
- Gating network selects specialists for different inputs
- Applied to small neural networks

**2010s - Deep Learning Era:**
- Eigen et al. (2013): Learning factored representations in deep learning
- Limited success due to training instability
- Difficult to balance expert loads

**2017 - Breakthrough at Scale:**
- Shazeer et al. (Google Brain): "Outrageously Large Neural Networks: The Sparsely-Gated Mixture-of-Experts Layer"
- 137B parameter model with 2048 experts
- Top-k gating: activate only k experts per token
- Load balancing loss to prevent expert collapse
- Achieved better perplexity than dense models at similar compute

**2021 - Switch Transformers:**
- Fedus et al. (Google): Switch Transformers
- Simplified to top-1 routing (one expert per token)
- 1.6T parameter model
- Demonstrated sparse models can be stable and efficient
- Better initialization and regularization strategies

**2022 - GLaM (Google):**
- 1.2T parameter MoE model
- 64 experts, top-2 routing
- Outperformed GPT-3 with 1/3 the energy for training
- Showed MoE can be more efficient than dense at scale

**2023 - Mixtral (Mistral AI):**
- Mixtral 8x7B: 8 experts, ~47B parameters, top-2 routing
- Open-source, production-ready MoE
- Matches/exceeds Llama 2 70B performance
- Active parameters per token: ~13B
- Demonstrated MoE is viable for open models

**2024 - DeepSeek-MoE:**
- Fine-grained expert specialization
- Shared experts + routed experts
- 145B total, 21B activated per token
- Strong performance on code and reasoning

**2024 - Grok (xAI):**
- 314B parameter MoE
- 8 experts, top-2 routing
- ~86B active parameters

### Theoretical Foundations

**Core MoE Equation:**
$$y = \sum_{i=1}^{N} G(x)_i \cdot E_i(x)$$

Where:
- $N$ = number of experts
- $G(x)$ = gating network output (routing weights)
- $E_i(x)$ = output of expert $i$
- $y$ = final output

**Top-k Sparse Gating:**
$$G(x)_i = \begin{cases}
\text{softmax}(\text{TopK}(W_g \cdot x))_i & \text{if } i \in \text{TopK} \\
0 & \text{otherwise}
\end{cases}$$

Only top-k experts are activated (typically k=1 or k=2).

**Advantages:**
1. **Scalability:** Add parameters without proportional compute increase
2. **Specialization:** Different experts learn different patterns
3. **Efficiency:** Sparse activation reduces inference cost
4. **Quality:** Can match dense models with fewer FLOPs

**Challenges:**
1. **Load balancing:** Preventing expert collapse
2. **Training stability:** Complex optimization landscape
3. **Memory:** All experts must fit in memory
4. **Communication:** Expert routing requires coordination

## 2. Mathematical Foundations

### Gating Mechanism

**1. Softmax Gating (Original):**
$$G(x) = \text{softmax}(W_g \cdot x)$$

Simple but can lead to load imbalance.

**2. Noisy Top-k Gating (Shazeer et al.):**
$$H(x)_i = (W_g \cdot x)_i + \epsilon \cdot \text{Softplus}((W_{\text{noise}} \cdot x)_i)$$
$$G(x) = \text{Softmax}(\text{TopK}(H(x), k))$$

Adds trainable noise for exploration during training.

**3. Switch Routing (Top-1):**
$$G(x)_i = \begin{cases}
1 & \text{if } i = \arg\max_j (W_g \cdot x)_j \\
0 & \text{otherwise}
\end{cases}$$

Simplified to single expert per token.

### Load Balancing

**Load Balancing Loss:**
$$L_{\text{balance}} = \alpha \cdot N \cdot \sum_{i=1}^{N} f_i \cdot P_i$$

Where:
- $f_i$ = fraction of tokens routed to expert $i$
- $P_i$ = average routing probability for expert $i$
- $\alpha$ = balancing coefficient (typically 0.01)
- $N$ = number of experts

Minimizes load imbalance across experts.

**Capacity Factor:**
$$\text{capacity}_i = \left\lceil \frac{k \cdot \text{tokens}}{N} \cdot \text{capacity\_factor} \right\rceil$$

Maximum tokens per expert. Typical capacity_factor = 1.25.

**Importance Loss (Alternative):**
$$L_{\text{importance}} = \text{CV}\left(\sum_{x} G(x)_i\right)^2$$

Minimizes coefficient of variation in expert usage.

### Expert Computation

**Standard FFN Expert:**
$$E_i(x) = W_{i,2} \cdot \text{GELU}(W_{i,1} \cdot x)$$

Each expert is a standard feed-forward network.

**Sparse Activation:**
- Total parameters: $N \times d_{\text{model}} \times d_{\text{ffn}} \times 2$
- Active parameters per token: $k \times d_{\text{model}} \times d_{\text{ffn}} \times 2$
- Activation ratio: $k / N$

For Mixtral (8 experts, top-2): activation ratio = 2/8 = 25%

### Efficiency Metrics

**FLOPs per Token:**
$$\text{FLOPs}_{\text{MoE}} = \text{FLOPs}_{\text{attention}} + k \cdot \text{FLOPs}_{\text{expert}}$$

Much less than:
$$\text{FLOPs}_{\text{dense}} = \text{FLOPs}_{\text{attention}} + N \cdot \text{FLOPs}_{\text{expert}}$$

**Memory Requirements:**
- Training: $O(N \cdot d_{\text{model}} \cdot d_{\text{ffn}})$ (all experts)
- Inference: Same (all experts loaded)
- Activation: $O(k \cdot d_{\text{model}} \cdot d_{\text{ffn}})$ (only active experts)

## 3. Implementation: Basic MoE Layer

In [ ]:
class Expert(nn.Module):
    """Single expert network (FFN)."""
    
    def __init__(self, input_dim: int, hidden_dim: int, dropout: float = 0.1):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, input_dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass through expert."""
        x = F.gelu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

class TopKRouter(nn.Module):
    """Top-k gating mechanism for routing tokens to experts."""
    
    def __init__(
        self,
        input_dim: int,
        num_experts: int,
        top_k: int = 2,
        noise_std: float = 1.0,
        use_noise: bool = True
    ):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.use_noise = use_noise
        self.noise_std = noise_std
        
        # Gating network
        self.gate = nn.Linear(input_dim, num_experts, bias=False)
        
        # Optional noise for exploration
        if use_noise:
            self.noise_gate = nn.Linear(input_dim, num_experts, bias=False)
    
    def forward(
        self,
        x: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Route tokens to experts.
        
        Args:
            x: Input tensor [batch_size, seq_len, input_dim]
        
        Returns:
            routing_weights: Weights for selected experts [batch, seq, top_k]
            selected_experts: Expert indices [batch, seq, top_k]
            load_balance_loss: Auxiliary loss for load balancing
        """
        batch_size, seq_len, _ = x.size()
        
        # Compute gate logits
        gate_logits = self.gate(x)  # [batch, seq, num_experts]
        
        # Add noise during training
        if self.training and self.use_noise:
            noise_logits = self.noise_gate(x)
            noise = torch.randn_like(gate_logits) * F.softplus(noise_logits) * self.noise_std
            gate_logits = gate_logits + noise
        
        # Select top-k experts
        top_k_logits, top_k_indices = torch.topk(
            gate_logits,
            k=min(self.top_k, self.num_experts),
            dim=-1
        )
        
        # Compute routing weights (softmax over top-k)
        routing_weights = F.softmax(top_k_logits, dim=-1)
        
        # Compute load balancing loss
        load_balance_loss = self._compute_load_balance_loss(
            gate_logits,
            top_k_indices
        )
        
        return routing_weights, top_k_indices, load_balance_loss
    
    def _compute_load_balance_loss(
        self,
        gate_logits: torch.Tensor,
        selected_experts: torch.Tensor
    ) -> torch.Tensor:
        """
        Compute load balancing loss.
        
        Encourages uniform distribution of tokens across experts.
        """
        # Compute routing probabilities
        routing_probs = F.softmax(gate_logits, dim=-1)  # [batch, seq, num_experts]
        
        # Average probability of routing to each expert
        mean_probs = routing_probs.mean(dim=(0, 1))  # [num_experts]
        
        # Fraction of tokens actually routed to each expert
        batch_size, seq_len, _ = gate_logits.size()
        expert_mask = torch.zeros(
            batch_size, seq_len, self.num_experts,
            device=gate_logits.device
        )
        expert_mask.scatter_(-1, selected_experts, 1)
        tokens_per_expert = expert_mask.sum(dim=(0, 1)) / (batch_size * seq_len * self.top_k)
        
        # Load balance loss: mean_probs * tokens_per_expert
        # When balanced, both should be ~1/num_experts
        load_balance_loss = self.num_experts * (mean_probs * tokens_per_expert).sum()
        
        return load_balance_loss

class MixtureOfExpertsLayer(nn.Module):
    """Complete MoE layer with routing and experts."""
    
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        num_experts: int = 8,
        top_k: int = 2,
        capacity_factor: float = 1.25,
        load_balance_weight: float = 0.01
    ):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_experts = num_experts
        self.top_k = top_k
        self.capacity_factor = capacity_factor
        self.load_balance_weight = load_balance_weight
        
        # Create experts
        self.experts = nn.ModuleList([
            Expert(input_dim, hidden_dim)
            for _ in range(num_experts)
        ])
        
        # Router
        self.router = TopKRouter(
            input_dim=input_dim,
            num_experts=num_experts,
            top_k=top_k
        )
        
        # Track statistics
        self.register_buffer('expert_usage', torch.zeros(num_experts))
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass through MoE layer.
        
        Args:
            x: Input tensor [batch_size, seq_len, input_dim]
        
        Returns:
            output: Mixed expert outputs [batch_size, seq_len, input_dim]
            aux_loss: Auxiliary losses (load balancing, etc.)
        """
        batch_size, seq_len, input_dim = x.size()
        
        # Get routing decisions
        routing_weights, selected_experts, load_balance_loss = self.router(x)
        
        # Flatten batch and sequence dimensions
        x_flat = x.view(-1, input_dim)  # [batch*seq, input_dim]
        selected_experts_flat = selected_experts.view(-1, self.top_k)  # [batch*seq, top_k]
        routing_weights_flat = routing_weights.view(-1, self.top_k)  # [batch*seq, top_k]
        
        # Initialize output
        output = torch.zeros_like(x_flat)
        
        # Route to experts
        for expert_idx in range(self.num_experts):
            # Find tokens routed to this expert
            expert_mask = (selected_experts_flat == expert_idx)
            token_indices = expert_mask.any(dim=-1).nonzero(as_tuple=True)[0]
            
            if len(token_indices) == 0:
                continue
            
            # Get tokens for this expert
            expert_input = x_flat[token_indices]
            
            # Process through expert
            expert_output = self.experts[expert_idx](expert_input)
            
            # Get weights for this expert
            # For each token, find where this expert appears in top-k
            expert_weights = torch.zeros(len(token_indices), device=x.device)
            for i, token_idx in enumerate(token_indices):
                expert_positions = (selected_experts_flat[token_idx] == expert_idx).nonzero(as_tuple=True)[0]
                if len(expert_positions) > 0:
                    expert_weights[i] = routing_weights_flat[token_idx, expert_positions].sum()
            
            # Weighted accumulation
            output[token_indices] += expert_weights.unsqueeze(-1) * expert_output
            
            # Track usage
            if self.training:
                self.expert_usage[expert_idx] += len(token_indices)
        
        # Reshape output
        output = output.view(batch_size, seq_len, input_dim)
        
        # Total auxiliary loss
        aux_loss = self.load_balance_weight * load_balance_loss
        
        return output, aux_loss
    
    def get_expert_statistics(self) -> Dict[str, torch.Tensor]:
        """Get statistics about expert usage."""
        total_usage = self.expert_usage.sum()
        if total_usage > 0:
            usage_pct = self.expert_usage / total_usage * 100
        else:
            usage_pct = torch.zeros_like(self.expert_usage)
        
        return {
            'expert_usage': self.expert_usage.clone(),
            'usage_percentage': usage_pct,
            'load_balance': usage_pct.std().item(),  # Lower is better
            'max_load': usage_pct.max().item(),
            'min_load': usage_pct.min().item()
        }
    
    def reset_statistics(self):
        """Reset expert usage statistics."""
        self.expert_usage.zero_()

print("Basic MoE layer implemented")

## 4. Visualizing MoE Behavior

In [ ]:
def visualize_moe_architecture():
    """Visualize MoE architecture and routing."""
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. MoE vs Dense comparison
    ax1 = axes[0, 0]
    
    model_sizes = ['7B', '13B', '47B (Mixtral)', '70B']
    params = [7, 13, 47, 70]  # Total parameters
    active_params = [7, 13, 13, 70]  # Active during inference
    is_moe = [False, False, True, False]
    
    x = np.arange(len(model_sizes))
    width = 0.35
    
    colors = ['coral' if moe else 'skyblue' for moe in is_moe]
    ax1.bar(x - width/2, params, width, label='Total Parameters', alpha=0.8, color=colors)
    ax1.bar(x + width/2, active_params, width, label='Active Parameters', alpha=0.8, color='green')
    
    ax1.set_ylabel('Parameters (Billions)')
    ax1.set_title('MoE vs Dense Models')
    ax1.set_xticks(x)
    ax1.set_xticklabels(model_sizes, rotation=15, ha='right')
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis='y')
    
    # 2. Expert routing distribution (simulated)
    ax2 = axes[0, 1]
    
    # Simulate expert usage over training
    np.random.seed(42)
    num_experts = 8
    steps = 100
    
    # Without load balancing (some experts dominate)
    usage_no_balance = np.random.dirichlet(np.array([10, 8, 6, 4, 2, 1, 0.5, 0.3]) * 10, steps)
    
    # With load balancing (more uniform)
    usage_balanced = np.random.dirichlet(np.ones(num_experts) * 5, steps)
    
    # Plot final distribution
    final_no_balance = usage_no_balance[-1] * 100
    final_balanced = usage_balanced[-1] * 100
    
    x_experts = np.arange(num_experts)
    width = 0.35
    
    ax2.bar(x_experts - width/2, final_no_balance, width, 
           label='Without Load Balancing', alpha=0.7, color='red')
    ax2.bar(x_experts + width/2, final_balanced, width,
           label='With Load Balancing', alpha=0.7, color='green')
    
    ax2.axhline(y=100/num_experts, color='black', linestyle='--', 
               label=f'Ideal ({100/num_experts:.1f}%)', alpha=0.5)
    ax2.set_xlabel('Expert ID')
    ax2.set_ylabel('Usage (%)')
    ax2.set_title('Expert Load Distribution')
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis='y')
    
    # 3. Training dynamics
    ax3 = axes[1, 0]
    
    # Simulate training loss and load balance over time
    training_steps = np.arange(1, 101)
    
    # Loss decreases for both
    loss_no_balance = 3.0 * np.exp(-training_steps / 40) + 1.2
    loss_balanced = 3.0 * np.exp(-training_steps / 35) + 1.0  # Better final loss
    
    ax3.plot(training_steps, loss_no_balance, label='Without Load Balancing', 
            linewidth=2, color='red')
    ax3.plot(training_steps, loss_balanced, label='With Load Balancing',
            linewidth=2, color='green')
    
    ax3.set_xlabel('Training Steps (x1000)')
    ax3.set_ylabel('Loss')
    ax3.set_title('Impact of Load Balancing on Training')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 4. Efficiency comparison
    ax4 = axes[1, 1]
    
    # FLOPs per token (normalized)
    models = ['Dense 13B', 'Mixtral 8x7B\n(47B total)', 'Dense 70B']
    flops_normalized = [1.0, 1.0, 5.4]  # Relative to Dense 13B
    performance = [100, 108, 112]  # Hypothetical benchmark score
    
    # Create efficiency plot
    colors_eff = ['skyblue', 'coral', 'purple']
    scatter = ax4.scatter(flops_normalized, performance, s=[500, 500, 500],
                         c=colors_eff, alpha=0.6)
    
    for i, model in enumerate(models):
        ax4.annotate(model, (flops_normalized[i], performance[i]),
                    xytext=(10, 10), textcoords='offset points',
                    fontsize=9, bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
    
    ax4.set_xlabel('Relative FLOPs per Token')
    ax4.set_ylabel('Performance Score')
    ax4.set_title('Efficiency: Performance vs Compute')
    ax4.grid(True, alpha=0.3)
    
    # Add efficiency frontier line
    ax4.plot([0, 6], [95, 115], 'k--', alpha=0.3, label='Ideal efficiency')
    ax4.legend()
    
    plt.tight_layout()
    plt.savefig('moe_architecture_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\nMixture of Experts Key Insights:")
    print("="*60)
    print("\n1. Parameter Efficiency:")
    print("   • Mixtral 8x7B: 47B total, 13B active per token")
    print("   • Matches/exceeds Dense 70B with ~5x fewer active FLOPs")
    print("\n2. Load Balancing is Critical:")
    print("   • Without: Some experts dominate, others unused")
    print("   • With: Uniform distribution, better performance")
    print("\n3. Training Trade-offs:")
    print("   • Memory: Must load all experts (higher than active params)")
    print("   • Compute: Only active experts used (lower than total params)")
    print("   • Communication: Routing requires coordination")

visualize_moe_architecture()
print("\nMoE visualization complete")

## 5. Fine-Tuning Strategies for MoE

In [ ]:
@dataclass
class MoEFineTuningConfig:
    """Configuration for MoE fine-tuning."""
    strategy: str  # 'all_experts', 'router_only', 'expert_subset', 'lora'
    freeze_router: bool = False
    freeze_experts: bool = False
    expert_dropout: float = 0.0  # Dropout experts during training
    load_balance_weight: float = 0.01
    lora_rank: Optional[int] = None  # If using LoRA on experts
    selected_experts: Optional[List[int]] = None  # For expert subset

class MoEFineTuner:
    """Fine-tuning manager for MoE models."""
    
    def __init__(
        self,
        model: nn.Module,
        config: MoEFineTuningConfig
    ):
        self.model = model
        self.config = config
        
        # Apply fine-tuning strategy
        self._configure_trainable_parameters()
    
    def _configure_trainable_parameters(self):
        """Set which parameters are trainable based on strategy."""
        strategy = self.config.strategy
        
        # First, freeze everything
        for param in self.model.parameters():
            param.requires_grad = False
        
        if strategy == 'all_experts':
            # Train all experts and routers
            for name, param in self.model.named_parameters():
                if 'moe' in name.lower() or 'expert' in name.lower():
                    param.requires_grad = True
        
        elif strategy == 'router_only':
            # Only train routing mechanisms
            for name, param in self.model.named_parameters():
                if 'router' in name.lower() or 'gate' in name.lower():
                    param.requires_grad = True
        
        elif strategy == 'expert_subset':
            # Train only selected experts
            if self.config.selected_experts:
                for expert_idx in self.config.selected_experts:
                    for name, param in self.model.named_parameters():
                        if f'expert.{expert_idx}' in name or f'experts.{expert_idx}' in name:
                            param.requires_grad = True
        
        elif strategy == 'lora':
            # Apply LoRA to experts (placeholder)
            print("LoRA fine-tuning for MoE experts...")
            # In practice, use LoRA library to add adapters to each expert
        
        # Apply freezing overrides
        if self.config.freeze_router:
            for name, param in self.model.named_parameters():
                if 'router' in name.lower() or 'gate' in name.lower():
                    param.requires_grad = False
        
        if self.config.freeze_experts:
            for name, param in self.model.named_parameters():
                if 'expert' in name.lower():
                    param.requires_grad = False
    
    def get_trainable_parameters(self) -> Dict[str, int]:
        """Get count of trainable parameters."""
        trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.model.parameters())
        
        return {
            'trainable': trainable,
            'total': total,
            'percentage': trainable / total * 100
        }
    
    def analyze_expert_specialization(
        self,
        data_loader: any,
        num_batches: int = 100
    ) -> Dict[int, Dict[str, float]]:
        """
        Analyze what each expert specializes in.
        
        Returns statistics about expert usage patterns.
        """
        self.model.eval()
        
        expert_stats = defaultdict(lambda: {
            'usage_count': 0,
            'avg_routing_weight': 0.0,
            'token_types': defaultdict(int)  # If we have token type info
        })
        
        with torch.no_grad():
            for batch_idx, batch in enumerate(data_loader):
                if batch_idx >= num_batches:
                    break
                
                # Forward pass (placeholder - actual implementation depends on model)
                # outputs, routing_info = self.model(batch, return_routing=True)
                pass
        
        return dict(expert_stats)

# Demonstrate fine-tuning strategies
def demonstrate_moe_finetuning_strategies():
    """Compare different MoE fine-tuning strategies."""
    
    strategies = {
        'Full Fine-Tuning': {
            'trainable_pct': 100.0,
            'memory_gb': 180,
            'speed': 1.0,
            'quality': 100
        },
        'Router Only': {
            'trainable_pct': 0.1,
            'memory_gb': 50,
            'speed': 3.5,
            'quality': 75
        },
        'Expert Subset (2/8)': {
            'trainable_pct': 25.0,
            'memory_gb': 90,
            'speed': 2.0,
            'quality': 88
        },
        'LoRA on Experts': {
            'trainable_pct': 1.5,
            'memory_gb': 55,
            'speed': 3.0,
            'quality': 92
        },
    }
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    names = list(strategies.keys())
    
    # 1. Trainable parameters
    ax1 = axes[0, 0]
    trainable = [s['trainable_pct'] for s in strategies.values()]
    colors = ['red', 'green', 'orange', 'blue']
    ax1.barh(names, trainable, color=colors, alpha=0.7)
    ax1.set_xlabel('Trainable Parameters (%)')
    ax1.set_title('Parameter Efficiency')
    ax1.set_xscale('log')
    ax1.grid(True, alpha=0.3, axis='x')
    
    # 2. Memory requirements
    ax2 = axes[0, 1]
    memory = [s['memory_gb'] for s in strategies.values()]
    ax2.barh(names, memory, color=colors, alpha=0.7)
    ax2.set_xlabel('GPU Memory (GB)')
    ax2.set_title('Memory Requirements')
    ax2.grid(True, alpha=0.3, axis='x')
    
    # 3. Training speed
    ax3 = axes[1, 0]
    speed = [s['speed'] for s in strategies.values()]
    ax3.barh(names, speed, color=colors, alpha=0.7)
    ax3.set_xlabel('Relative Training Speed')
    ax3.set_title('Training Speed (higher is faster)')
    ax3.grid(True, alpha=0.3, axis='x')
    
    # 4. Quality vs efficiency
    ax4 = axes[1, 1]
    quality = [s['quality'] for s in strategies.values()]
    efficiency = [100 / s['memory_gb'] * s['speed'] for s in strategies.values()]
    
    scatter = ax4.scatter(efficiency, quality, s=[500]*len(names), 
                         c=colors, alpha=0.6)
    
    for i, name in enumerate(names):
        ax4.annotate(name, (efficiency[i], quality[i]),
                    xytext=(5, 5), textcoords='offset points',
                    fontsize=8)
    
    ax4.set_xlabel('Efficiency Score (speed / memory)')
    ax4.set_ylabel('Model Quality')
    ax4.set_title('Quality vs Efficiency Trade-off')
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('moe_finetuning_strategies.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\nMoE Fine-Tuning Strategy Recommendations:")
    print("="*70)
    print("\n1. Router Only:")
    print("   Best for: Quick adaptation, limited compute")
    print("   Trade-off: Lower quality, but very fast and memory-efficient")
    print("\n2. Expert Subset:")
    print("   Best for: Domain-specific fine-tuning")
    print("   Trade-off: Balanced quality and efficiency")
    print("\n3. LoRA on Experts:")
    print("   Best for: Parameter-efficient fine-tuning")
   print("   Trade-off: High quality with low memory overhead")
    print("\n4. Full Fine-Tuning:")
    print("   Best for: Maximum quality, sufficient resources")
    print("   Trade-off: Highest memory and compute cost")

demonstrate_moe_finetuning_strategies()
print("\nMoE fine-tuning strategies demonstrated")

## 6. When to Use MoE

### Ideal Use Cases

**1. Scaling Up Quality:**
- Need better performance than dense model
- Have sufficient memory for all experts
- Inference latency is less critical than throughput

**2. Multi-Domain Applications:**
- Model serves diverse tasks (code, math, writing)
- Different experts can specialize
- Benefit from task-specific optimization

**3. Batch Inference:**
- Large batch sizes (>16)
- Can amortize routing overhead
- Memory for all experts available

**4. Research & Experimentation:**
- Exploring architectural innovations
- Studying expert specialization
- Not constrained by deployment complexity

### When NOT to Use MoE

**1. Memory Constrained:**
- Cannot fit all experts in memory
- Consumer GPUs (limited VRAM)
- Edge devices

**2. Low Latency Required:**
- Real-time applications
- Interactive chat (single-user)
- Routing overhead matters

**3. Small Scale:**
- Models <7B parameters
- Limited training data
- Simple, focused tasks

**4. Simple Deployment:**
- Need easy deployment
- Limited infrastructure
- Prefer dense model simplicity

### Decision Matrix

| Factor | Dense Better | MoE Better |
|--------|--------------|------------|
| **Memory** | Limited (<80GB) | Ample (>160GB) |
| **Batch Size** | Small (1-8) | Large (>16) |
| **Task Diversity** | Single domain | Multi-domain |
| **Latency** | Critical | Flexible |
| **Quality Need** | Good enough | Maximize |
| **Deployment** | Simple | Can handle complexity |

### Practical Recommendations

**Choose MoE if:**
- You want 70B-class quality with 13B-class inference cost
- Serving many requests simultaneously
- Multi-domain application (code + text + math)
- Have H100/A100 GPUs with sufficient memory

**Choose Dense if:**
- Single-user or low-concurrency
- Memory constrained (<80GB)
- Need lowest possible latency
- Simpler deployment preferred

**Example Scenarios:**

**Scenario 1: Production API (High Traffic)**
- Use: Mixtral 8x7B
- Why: Batch inference, good throughput, quality/cost balance

**Scenario 2: Interactive Chatbot (Single User)**
- Use: Llama 3 8B (dense)
- Why: Lower latency, simpler deployment, sufficient quality

**Scenario 3: Code + Text Assistant**
- Use: DeepSeek-MoE or Mixtral
- Why: Experts can specialize in code vs text

**Scenario 4: Edge Device**
- Use: Quantized dense model
- Why: Memory constraints rule out MoE

## 7. Best Practices for MoE Fine-Tuning

### Training Recommendations

**1. Load Balancing:**
- Start with load_balance_weight = 0.01
- Increase if experts are imbalanced (std > 20%)
- Decrease if performance degrades
- Monitor expert usage regularly

**2. Learning Rates:**
- Router: 1-2x base LR (learns faster)
- Experts: 0.5-1x base LR (more stable)
- Use separate optimizers if needed

**3. Initialization:**
- Random: Can lead to expert collapse
- Pre-trained: Start from existing MoE weights
- Upcycling: Convert dense model to MoE

**4. Gradient Handling:**
- Use gradient clipping (max_norm = 1.0)
- Watch for expert gradient magnitudes
- Balance expert vs. router gradients

### Memory Optimization

**1. Expert Parallelism:**
- Distribute experts across GPUs
- Each GPU holds subset of experts
- Communication for routing

**2. Gradient Checkpointing:**
- Essential for large MoE models
- Trade compute for memory
- Can reduce memory by 50%+

**3. Mixed Precision:**
- FP16/BF16 for experts
- FP32 for router (stability)
- Can reduce memory by 40%

**4. LoRA on Experts:**
- r=16-32 per expert
- Only adapters trainable
- 95%+ memory savings

### Evaluation & Monitoring

**Key Metrics:**
1. **Expert Load Balance:** std(usage) < 20%
2. **Router Entropy:** Higher = better exploration
3. **Expert Specialization:** Track task-expert correlations
4. **Aux Loss:** Should decrease over training
5. **Dead Experts:** Experts with <1% usage

**Monitoring Dashboard:**
- Expert usage heatmap
- Load balance over time
- Per-expert performance
- Routing entropy
- Gradient norms

### Common Pitfalls

**1. Expert Collapse:**
- Symptom: 1-2 experts handle all tokens
- Fix: Increase load_balance_weight
- Prevention: Monitor from start

**2. Routing Instability:**
- Symptom: Expert assignments fluctuate wildly
- Fix: Lower router learning rate
- Prevention: Gradual warmup

**3. Memory OOM:**
- Symptom: Out of memory during forward pass
- Fix: Reduce batch size, use expert parallelism
- Prevention: Calculate memory requirements upfront

**4. Poor Specialization:**
- Symptom: Experts don't learn distinct patterns
- Fix: Use diverse training data, adjust capacity
- Prevention: Analyze expert-task correlations

### Fine-Tuning Checklist

**Pre-Training:**
- [ ] Calculate memory requirements
- [ ] Choose fine-tuning strategy
- [ ] Set up monitoring
- [ ] Prepare diverse training data

**During Training:**
- [ ] Monitor expert usage every 100 steps
- [ ] Check for dead experts
- [ ] Track load balance metric
- [ ] Validate on diverse tasks

**Post-Training:**
- [ ] Analyze expert specialization
- [ ] Measure inference efficiency
- [ ] Compare to dense baseline
- [ ] Document expert behaviors

## 8. Real-World MoE Models

### Mixtral 8x7B (Mistral AI, 2023)

**Architecture:**
- 8 experts per layer
- Top-2 routing
- ~47B total parameters
- ~13B active per token

**Performance:**
- Matches/exceeds Llama 2 70B
- 6x faster inference than Llama 2 70B
- MMLU: 70.6% (vs 68.9% for Llama 2 70B)

**Fine-Tuning:**
- Supports standard fine-tuning
- LoRA works well (r=16-32)
- Can fine-tune specific experts

### GPT-4 (OpenAI, 2023)

**Rumored Architecture:**
- ~1.8T total parameters
- 16 experts (unconfirmed)
- ~220B active per forward pass
- Top-2 routing

**Characteristics:**
- Strong multi-domain performance
- Likely expert specialization (code, math, etc.)
- Expensive inference (reflects MoE costs)

### DeepSeek-MoE (2024)

**Architecture:**
- 145B total parameters
- Fine-grained experts (more, smaller)
- Shared experts + routed experts
- ~21B active per token

**Innovations:**
- Hybrid shared/routed approach
- Better expert specialization
- Strong on code and math

### Performance Benchmarks

**MMLU (General Knowledge):**
- Mixtral 8x7B: 70.6%
- Llama 2 70B: 68.9%
- GPT-4: 86.4%

**HumanEval (Code):**
- Mixtral 8x7B: 40.2%
- DeepSeek-MoE: 45.0%
- GPT-4: 67.0%

**Inference Efficiency:**
- Mixtral 8x7B: ~6x faster than Llama 70B (same quality)
- Memory: All experts must be loaded
- Throughput: Excellent with batching

### Training Costs

**Mixtral 8x7B:**
- Training: ~$3-4M (estimated)
- Data: Undisclosed, likely 1-2T tokens
- Hardware: H100 cluster

**Fine-Tuning Costs:**
- Full fine-tune: $5K-20K (depending on data)
- LoRA: $500-2K
- Router only: $100-500

Much cheaper than training from scratch!

## Summary

Mixture of Experts enables scaling model capacity without proportional compute increase:

**Key Concepts:**
1. **Sparse Activation:** Only k of N experts active per token
2. **Router:** Learned gating network selects experts
3. **Load Balancing:** Critical for preventing expert collapse
4. **Specialization:** Experts learn different patterns/domains

**Advantages:**
- Scale to higher quality without proportional FLOPs
- Expert specialization for multi-domain tasks
- Better throughput with batching
- Can match larger dense models

**Trade-offs:**
- All experts must fit in memory
- Routing overhead (minor but exists)
- More complex training
- Load balancing required

**Fine-Tuning Strategies:**
1. **Router Only:** Fast, memory-efficient, lower quality
2. **Expert Subset:** Balanced approach
3. **LoRA on Experts:** High quality, parameter-efficient
4. **Full Fine-Tuning:** Maximum quality, highest cost

**When to Use:**
- Need 70B-class quality with 13B-class inference
- Multi-domain applications
- Batch inference scenarios
- Sufficient memory (>160GB)

**When to Avoid:**
- Memory constrained (<80GB)
- Low latency critical
- Simple deployment preferred
- Small scale applications

**Best Practices:**
- Monitor expert usage from the start
- Use load balancing loss (weight ~0.01)
- Consider LoRA for parameter efficiency
- Analyze expert specialization patterns
- Compare against dense baselines

MoE represents a promising direction for efficient scaling, exemplified by models like Mixtral 8x7B matching Llama 2 70B performance with significantly faster inference.